In [30]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
load_dotenv()

True

In [31]:
class Quad(TypedDict):
    a:int
    b:int
    c:int
    equation:str
    discriminant:float
    result:str

In [32]:
def show_equation(state:Quad)->Quad:
    equation=f'{state["a"]}x^2+{state["b"]}x+{state["c"]}'
    return {**state, 'equation': equation}

In [33]:
def calculate_discriminant(state:Quad)->Quad:
    discriminant=state['b']**2-(4*state['a']*state['c'])
    return {**state, 'discriminant': discriminant}

In [34]:
def real_roots(state:Quad)->Quad:
    root1=-state["b"]+(state["discriminant"]**0.5)/(2*state["a"])
    root2=-state["b"]-(state["discriminant"]**0.5)/(2*state["a"])
    return {**state, 'result': f'The roots are real and distinct: {root1} and {root2}'}

In [35]:
def repeated_roots(state:Quad)->Quad:
    root=-state["b"]/(2*state["a"])
    return {**state, 'result': f'The roots are real and repeated: {root}'}

In [36]:
def no_real_roots(state:Quad)->Quad:
    return {**state, 'result': 'The roots are imaginary and not real.'}

In [37]:

from typing import Literal


def check_condition(state:Quad)->Literal['real_roots','repeated_roots','no_real_roots']:
    if state['discriminant']>0:
        return 'real_roots'
    elif state['discriminant']==0:
        return 'repeated_roots'
    else:
        return 'no_real_roots'

In [38]:
graph=StateGraph(Quad)
graph.add_node('show_equation',show_equation)
graph.add_node('calculate_discriminant',calculate_discriminant)

graph.add_node('real_roots',real_roots)
graph.add_node('repeated_roots',repeated_roots)
graph.add_node('no_real_roots',no_real_roots)

graph.add_edge(START,'show_equation')
graph.add_edge('show_equation','calculate_discriminant')
graph.add_conditional_edges('calculate_discriminant',check_condition)

graph.add_edge('real_roots',END)
graph.add_edge('repeated_roots',END)
graph.add_edge('no_real_roots',END)

workflow=graph.compile()


In [45]:
initial_state={'a': 4, 'b': 2, 'c': 4}
result=workflow.invoke(initial_state)
print(result)

{'a': 4, 'b': 2, 'c': 4, 'equation': '4x^2+2x+4', 'discriminant': -60, 'result': 'The roots are imaginary and not real.'}
